# 半全场历史数据验证

本笔记本只读取本地 `.codex/soccer-predict` 数据、模型注册表和固定赛季评估 artifact。它复核数据哈希、逐场指标、联赛差异、陌生球队 fallback 与日职特殊赛制；不会把历史概率覆盖率解释为下注胜率或 ROI。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Markdown, display

repo = Path.cwd().resolve()
if not (repo / 'scripts').is_dir():
    raise RuntimeError('请从 Football-predictions 仓库根目录执行本笔记本')
sys.path.insert(0, str(repo))
from scripts import history_importer, htft_holdout_evaluator, league_model_manager, htft_ranker

runtime = repo / '.codex' / 'soccer-predict'
dataset_dir = runtime / 'datasets' / 'league-history'
evaluation_path = runtime / 'evaluations' / 'htft-fixed-seasons.json'
model_dir = runtime / 'models' / 'league-history'
for required in (dataset_dir / 'manifest.json', evaluation_path, model_dir / 'registry.json'):
    if not required.is_file():
        raise FileNotFoundError(f'缺少本地 artifact：{required}')

In [2]:
manifest = history_importer.validate_bundle(dataset_dir)
evaluation = json.loads(evaluation_path.read_text(encoding='utf-8'))
htft_holdout_evaluator.validate_evaluation(evaluation, dataset_dir=dataset_dir)
registry = league_model_manager.load_registry(model_dir)
assert registry['dataset_manifest_hash'] == manifest['bundle_hash']
assert evaluation['dataset']['manifest_bundle_hash'] == manifest['bundle_hash']
assert evaluation['promotion']['final_selector_untouched'] is False
assert evaluation['promotion']['end_to_end_promotion_eligible'] is False
display(Markdown(
    f"**语义验证通过**  数据包 `{manifest['bundle_hash']}`；"
    f"评估 `{evaluation['evaluation_hash']}`；注册表 `{registry['registry_hash']}`。"
))

**语义验证通过**  数据包 `sha256:66fa77f06227492cc61a68c3414312116d200d603c7f351bd1dd26a05a5c2346`；评估 `sha256:c3055c821eee5db04936dad430fa3ed792646c2d177776cf156cf8b6928f9332`；注册表 `sha256:ba42c65dabcdd5b60b47019479773f70f291cb2515b2124ba08939d53296283d`。

In [3]:
lineage_rows = []
for league in manifest['leagues']:
    special = sum(
        count
        for by_regime in league['competition_regimes'].values()
        for regime, count in by_regime.items()
        if regime != 'regular'
    )
    registered = next(x for x in registry['leagues'] if x['league_key'] == league['league_key'])
    lineage_rows.append({
        '联赛': league['league'],
        'league_key': league['league_key'],
        '源比赛': league['rows'],
        '生产训练': registered['training_rows'],
        '特殊赛制排除': registered['excluded_training_rows'],
        '训练截止': registered['training_cutoff'],
        '特殊赛制源记录': special,
    })
lineage = pd.DataFrame(lineage_rows)
assert int(lineage['源比赛'].sum()) == 9211
display(lineage)

,联赛,league_key,源比赛,生产训练,特殊赛制排除,训练截止,特殊赛制源记录
0,巴甲,brazil_serie_a,2479,2479,0,2026-07-26,0
1,日职,japan_j1,2238,2058,180,2025-12-06,180
2,挪超,norway_eliteserien,1554,1554,0,2026-07-27,0
3,美职联,usa_mls,2940,2940,0,2026-07-26,0


In [4]:
split_rows = []
for split_id, summary in evaluation['summary']['by_split'].items():
    metrics = summary['model_only']['overall']['metrics']
    gate = summary['model_only']['overall']['pair_mass_gate']
    baseline = summary['league_empirical_frequency_baseline']['metrics']
    split_rows.append({
        'split': split_id,
        'n': metrics['sample_count'],
        'log_loss': metrics['nine_class_log_loss'],
        'Brier': metrics['nine_class_brier'],
        'Top1': metrics['top_one_accuracy'],
        'Top2': metrics['top_two_accuracy'],
        '频率基线 log_loss': baseline['nine_class_log_loss'],
        '0.46覆盖': gate['coverage'],
        '覆盖内Top2': gate['hit_rate_when_covered'],
    })
split_table = pd.DataFrame(split_rows).sort_values('split')
display(split_table.round(5))

,split,n,log_loss,Brier,Top1,Top2,频率基线 log_loss,0.46覆盖,覆盖内Top2
0,fixed_holdout_2025,1510,1.92022,0.82267,0.30066,0.48477,1.94199,0.33709,0.55206
1,shadow_2026,566,1.90416,0.81159,0.33569,0.50707,1.92434,0.43993,0.56627
2,validation_2024,1493,1.95535,0.83465,0.28064,0.44273,1.96298,0.40388,0.46766


In [5]:
league_gate_rows = []
for league in evaluation['leagues']:
    split = next(x for x in league['splits'] if x['test_season'] == 2025)
    gate = split['model_only']['overall']['pair_mass_gate']
    evidence = htft_ranker._league_pair_gate_evidence(league['league_key'])
    league_gate_rows.append({
        '联赛': league['league'],
        '2025比赛': split['test_match_count'],
        '0.46覆盖场': gate['covered_count'],
        '命中': gate['hit_count'],
        '覆盖内Top2': gate['hit_rate_when_covered'],
        'Wilson 95%下界': evidence['wilson_95_lower_bound'],
        'ranker状态': evidence['status'],
    })
league_gate = pd.DataFrame(league_gate_rows)
assert not any(x['production_confidence_eligible'] for x in [
    htft_ranker._league_pair_gate_evidence(key)
    for key in htft_ranker.LEAGUE_PAIR_GATE_EVIDENCE
])
display(league_gate.round(5))

,联赛,2025比赛,0.46覆盖场,命中,覆盖内Top2,Wilson 95%下界,ranker状态
0,巴甲,380,125,72,0.57600,0.48837,league_gate_lower_bound_not_above_chance
1,日职,380,66,32,0.48485,0.36847,competition_regime_shift_unconfirmed
2,挪超,240,110,63,0.57273,0.47937,league_gate_lower_bound_not_above_chance
3,美职联,510,208,114,0.54808,0.48018,league_gate_lower_bound_not_above_chance


In [6]:
cohorts = evaluation['summary']['by_split']['fixed_holdout_2025']['model_only']
fallback_rows = []
for key, label in [('known_teams', '已知球队'), ('league_average_fallback', '联赛均值 fallback')]:
    metrics = cohorts[key]['metrics']
    fallback_rows.append({
        '队伍可用性': label,
        'n': metrics['sample_count'],
        'log_loss': metrics['nine_class_log_loss'],
        'Brier': metrics['nine_class_brier'],
        'Top1': metrics['top_one_accuracy'],
        'Top2': metrics['top_two_accuracy'],
    })
fallback_table = pd.DataFrame(fallback_rows)
assert fallback_table['n'].sum() == 1510
display(fallback_table.round(5))
display(Markdown('注册模型默认 `unknown_team_policy=error`；headline 中的 140 场 fallback 只为完整评估，不会在生产中静默启用。'))

,队伍可用性,n,log_loss,Brier,Top1,Top2
0,已知球队,1370,1.91190,0.82024,0.30511,0.49416
1,联赛均值 fallback,140,2.00162,0.84652,0.25714,0.39286


注册模型默认 `unknown_team_policy=error`；headline 中的 140 场 fallback 只为完整评估，不会在生产中静默启用。

In [7]:
paired = evaluation['summary']['by_split']['fixed_holdout_2025']['model_minus_empirical_baseline']
bootstrap_rows = []
for key, label in [('nine_class_log_loss', 'log loss'), ('nine_class_brier', 'Brier')]:
    item = paired[key]
    bootstrap_rows.append({
        '指标': label,
        '模型-基线': item['mean_delta'],
        '95%下界': item['ci95_low'],
        '95%上界': item['ci95_high'],
    })
display(pd.DataFrame(bootstrap_rows).round(5))

,指标,模型-基线,95%下界,95%上界
0,log loss,-0.02177,-0.03260,-0.01049
1,Brier,-0.00893,-0.01277,-0.00524


In [8]:
japan_eval = next(x for x in evaluation['leagues'] if x['league_key'] == 'japan_j1')
japan_shadow = next(x for x in japan_eval['splits'] if x['test_season'] == 2026)
japan_registry = next(x for x in registry['leagues'] if x['league_key'] == 'japan_j1')
assert japan_shadow['test_match_count'] == 0
assert japan_shadow['test_competition_regime_counts'] == {}
assert japan_shadow['excluded_test_match_count'] == 180
assert japan_shadow['excluded_test_competition_regime_counts'] == {'2026_vision_regional': 180}
assert japan_registry['competition_regime_policy']['excluded_regime_counts'] == {'2026_vision_regional': 180}
display(Markdown(
    '**日职赛制保护已生效：** 2026 特殊地区阶段 180 场只保留排除计数，'
    '不进入常规 J1 注册模型训练或正式 shadow 指标；新赛季仍需新的前向样本确认。'
))

**日职赛制保护已生效：** 2026 特殊地区阶段 180 场只保留排除计数，不进入常规 J1 注册模型训练或正式 shadow 指标；新赛季仍需新的前向样本确认。

## 结论

锁定模型相对训练期联赛频率基线改善了九分类 proper scoring rules，但最终 Top-2 选择器已看过 2025/2026，且各联赛 0.46 覆盖门槛的 Wilson 下界都未高于 50%。因此当前半全场输出保持 observation-only，不能把 55.21% 当成未来单注胜率，也不能迁移到未训练的韩K联。